# 演習2. データ競合 ―― 守らないと壊れる

## シーン

演習1で、スレッドを分ければ速くなることを見ました。
しかし、分けた相手と**同じデータを触る**と、話は一気にややこしくなります。

C++問題集の**問9**で `std::ref` を使って1つのキューを共有し、
解答には `std::lock_guard` が出てきましたが、
「**なぜ必要なのか**」は実際に壊してみないと実感できません。

このノートでは、わざと保護しないプログラムを走らせて**壊れるところを見ます**。

**このノートを終えると、「この変数は守らないと危ない」を自分で判断できるようになります。**

## 2-1. 2つのスレッドで1つのカウンタを増やす

2つのスレッドが、同じ変数 `counter` を 1,000,000 回ずつ増やします。
当然、最後は **2,000,000** になるはずです。

実行前に予測してください。本当に 2,000,000 になるでしょうか。

In [ ]:
%%writefile ex02a.cpp
#include <iostream>
#include <thread>

long counter = 0;                 // 2つのスレッドが共有する変数

void add_many() {
    for (int i = 0; i < 1000000; i++) {
        counter++;                // ← 保護していない！
    }
}

int main() {
    std::thread t1(add_many);
    std::thread t2(add_many);
    t1.join();
    t2.join();

    std::cout << "counter = " << counter
              << "   (期待値 2000000)\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex02a.cpp -o ex02a
# 5回続けて実行してみる
!for i in 1 2 3 4 5; do ./ex02a; done

**5回とも違う値**、しかもどれも 2,000,000 に届かなかったはずです。

### なぜ壊れるのか

`counter++` は1行ですが、CPU から見ると**3ステップ**の作業です。

- **① 読む**：メモリから `counter` の値を取ってくる
- **② 足す**：取ってきた値に 1 を足す
- **③ 書く**：結果をメモリに書き戻す

2つのスレッドがこの3ステップを**同時に**始めると、こうなります。

```
スレッドA: ① 読む(=100) ② 足す(=101)              ③ 書く(101)
スレッドB:              ① 読む(=100) ② 足す(=101) ③ 書く(101)
                                                   ↑ 2回足したのに 101。1回分が消えた
```

これが**データ競合（race condition）**です。
やっかいなのは、**毎回起きるとは限らない**こと。タイミング次第なので、
「たまに動く」「手元では再現しない」という最悪の形のバグになります。

## 2-2.【コラム】カウンタ1個なら `std::atomic` で直せる

「①読む ②足す ③書く」を**分割できない1つの操作**にしてくれるのが `std::atomic` です。
`long` を `std::atomic<long>` に変えるだけで直ります。

> **これは寄り道です。** 読み飛ばして 2-3 に進んでも構いません。
> ここで扱うのは、(1) 競合の直し方は1つではないと知るため、
> (2) すぐ次の 2-3 で「**`atomic` では直せないものがある**」ことを見せ、
> `mutex` がなぜ必要かにつなげるため、の2つの理由からです。
>
> `atomic` が本当に効くのは、カウンタや**停止フラグ**のように
> 「1つの変数を、読むか書くかするだけ」の場合に限られます（演習9で再登場します）。

In [ ]:
%%writefile ex02b.cpp
#include <iostream>
#include <thread>
#include <atomic>

std::atomic<long> counter(0);     // ← long から atomic<long> に変えただけ

void add_many() {
    for (int i = 0; i < 1000000; i++) {
        counter++;                // これで「分割できない1つの操作」になる
    }
}

int main() {
    std::thread t1(add_many);
    std::thread t2(add_many);
    t1.join();
    t2.join();

    std::cout << "counter = " << counter
              << "   (期待値 2000000)\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex02b.cpp -o ex02b
!for i in 1 2 3 4 5; do ./ex02b; done

今度は5回とも 2,000,000 になったはずです。

## 2-3. `atomic` では直せないものもある

では `atomic` があれば十分かというと、そうではありません。
次は、2つのスレッドが**保護なしで同じ `std::queue` から取り出す**プログラムです。
キューには 200,000 個入れてあるので、2つのスレッドが取り出した個数の**合計は 200,000** になるはずです。

> **注意**：このプログラムは**異常終了することがあります**。それも競合の症状の1つです。
> エラーが出たら、それ自体が観察結果です。何度か実行してみてください。

In [ ]:
%%writefile ex02c.cpp
#include <iostream>
#include <thread>
#include <queue>

std::queue<int> q;                // 保護なしで共有するキュー
long taken[2] = {0, 0};           // 各スレッドが取り出した個数

void worker(int id) {
    while (true) {
        if (q.empty()) return;    // ← 空か確かめて…
        q.pop();                  // ← 取り出す（この2行の「すきま」が危ない）
        taken[id]++;
    }
}

int main() {
    for (int i = 0; i < 200000; i++) q.push(i);

    std::thread t1(worker, 0);
    std::thread t2(worker, 1);
    t1.join();
    t2.join();

    std::cout << "スレッド0 = " << taken[0]
              << " / スレッド1 = " << taken[1]
              << " / 合計 = " << taken[0] + taken[1]
              << "   (期待値 200000)\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex02c.cpp -o ex02c
# 異常終了しても続けたいので、失敗しても次に進むようにしておく
!for i in 1 2 3; do ./ex02c || echo "  → 異常終了しました（これも競合の症状）"; done

合計が 200,000 にならなかったり、異常終了したりしたはずです。

危ないのは、`if (q.empty())` と `q.pop()` の**すきま**です。

```
スレッドA: if(q.empty()) → 残り1個ある。よし取ろう
スレッドB:                  if(q.empty()) → 残り1個ある。よし取ろう
スレッドA:                                  q.pop()  ← 最後の1個を取った
スレッドB:                                            q.pop()  ← 空なのに取った！
```

`std::queue` を `atomic` にすることはできません。
`atomic` が守れるのは「1つの変数への1回の読み書き」だけで、
**「確かめてから取り出す」のような複数ステップのまとまり**は守れないからです。

こういうときに使うのが **`mutex`** です。次の**演習3**で扱います。

## 発展課題

1. `ex02a.cpp` のループ回数を 1,000,000 から 1,000 に減らすと、
   結果はどうなると思いますか。予測してから確かめてください。
   そこから「競合バグの見つけにくさ」について何が言えるでしょうか。

2. 次の3つのグローバル変数は、どれも複数のスレッドが動いている間に存在します。
   **競合するのはどれで、しないのはどれか**、理由とともに答えてください。

   | 変数 | 使われ方 |
   |---|---|
   | `int index` | スレッドA だけが増やす。他のスレッドは触らない |
   | `bool stop` | スレッドA が書き、スレッドB が読む |
   | `Config conf` | スレッドを起動する**前**に `main` が設定し、以後は全スレッドが読むだけ |

3. `ex02c.cpp` の `taken[0]` と `taken[1]` は、2つのスレッドが**別々の要素**に書き込んでいます。
   これは競合しているでしょうか、していないでしょうか。理由も考えてください。

4. 共有するキューを守るクラスを作ったとして、`push` と `pop` は保護したけれど、
   「いま何個入っているか」を返す `size()` だけは保護しなかったとします。
   これは問題になるでしょうか。どういう場面で困るかを考えてください。